# B2.3 - Teacher labels and plain-English rewrites (FinSight, M5)

Runs a Qwen teacher (AWQ 4-bit) with **vLLM** over the corpus risks in the private dataset
`finsight-teacher-risks` (`risks.jsonl`: `risk_id`, `company`, `title`, `body`). Inference only;
nothing is trained here. One JSON answer per risk: `category` (one of 10), `seriousness_1to5`,
`hard_fact`, `simple` (<= 60 words), `numbers_copied`. Thinking mode is off.

- **Order:** smoke (50) -> pilot (500) -> full. Answers are appended to
  `/kaggle/working/teacher/raw.jsonl` every 100 risks; attach an earlier `raw.jsonl` as a dataset
  and the run skips what is done.
- **Filters run on the laptop**, not here (`python -m finsight.risks.teacher_data filter`), with
  the same checks the student's rewrites must pass.
- GPU: T4 (16 GB, fp16 compute; AWQ kernels work on Turing). With T4 x2 the model is split over
  both GPUs (more room for the KV cache). Record the GPU hours in PROGRESS.md.
- Teacher outputs are AI labels: every row the laptop writes carries `label_source`.


In [ ]:
# ---- parameters ------------------------------------------------------------------------------
MODEL = "Qwen/Qwen3-14B-AWQ"  # Apache-2.0; fallback "Qwen/Qwen2.5-14B-Instruct-AWQ"
DATASET = "finsight-teacher-risks"
LIMIT = 50  # smoke 50 | pilot 500 | full None
EVERY = 100  # checkpoint interval (risks per vLLM batch)
MAX_MODEL_LEN = 4096
MAX_TOKENS = 400
OUT_DIR = "/kaggle/working/teacher"


In [ ]:
# vLLM 0.8.5 still has the V0 engine (xformers attention), which runs on a T4 (sm75).
# Newer releases may need the V1 engine; check the vLLM docs before changing the pin.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.8.5.post1"], check=True)


In [ ]:
"""Resumable teacher run (B03 §3.4): raw answers appended to a JSONL checkpoint in batches.

Standard library only: ``scripts/make_teacher_notebook.py`` copies this file into the Kaggle
notebook, so the loop tested here is the loop that runs on the GPU. A rerun skips every
``risk_id`` already in the output file (the resume flag is simply running it again).
"""

from __future__ import annotations

import json
import os
from collections.abc import Callable, Iterable, Iterator
from pathlib import Path
from typing import Any

Generate = Callable[[list[list[dict[str, str]]]], list[str]]  # chats in, raw texts out


def read_jsonl(path: Path) -> Iterator[dict[str, Any]]:
    """Rows of a JSON Lines file (nothing if the file is missing)."""
    if not path.exists():
        return
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue  # a line cut short by a stopped session; that risk is redone


def done_ids(path: Path) -> set[str]:
    """Risk ids already answered, so a resumed run skips them."""
    return {str(row["risk_id"]) for row in read_jsonl(path) if "risk_id" in row}


def _append(path: Path, rows: list[dict[str, Any]]) -> None:
    cut = False
    if path.exists() and path.stat().st_size > 0:
        with path.open("rb") as f:
            f.seek(-1, os.SEEK_END)
            cut = f.read(1) != b"\n"
    with path.open("a", encoding="utf-8") as f:
        if cut:  # keep the stopped session's half line on its own, so it stays unreadable
            f.write("\n")
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


def run_teacher(
    items: Iterable[dict[str, Any]],
    generate: Generate,
    out_path: Path,
    *,
    every: int = 100,
    limit: int | None = None,
    meta: dict[str, Any] | None = None,
) -> dict[str, int]:
    """Send each item's ``messages`` to ``generate`` in batches of ``every`` and checkpoint.

    Args:
        items: dicts with ``risk_id`` and ``messages`` (chat format).
        generate: one call per batch; returns one raw text per chat, in order.
        out_path: JSONL checkpoint; rows are ``{risk_id, raw, **meta}``.
        every: batch size, and so the checkpoint interval.
        limit: stop after this many new items (pilot and smoke runs).
        meta: copied into every row (model, prompt_version).

    Returns:
        ``{"skipped": already done, "written": new rows}``.
    """
    out_path.parent.mkdir(parents=True, exist_ok=True)
    done = done_ids(out_path)
    skipped = written = 0
    batch: list[dict[str, Any]] = []

    def flush() -> None:
        nonlocal written
        if not batch:
            return
        raws = generate([it["messages"] for it in batch])
        if len(raws) != len(batch):
            raise ValueError(f"generate returned {len(raws)} answers for {len(batch)} chats")
        _append(
            out_path,
            [
                {"risk_id": it["risk_id"], "raw": r, **(meta or {})}
                for it, r in zip(batch, raws, strict=True)
            ],
        )
        written += len(batch)
        print(f"checkpoint: {written} new rows in {out_path.name}", flush=True)
        batch.clear()

    for item in items:
        rid = str(item["risk_id"])
        if rid in done:
            skipped += 1
            continue
        if limit is not None and written + len(batch) >= limit:
            break
        done.add(rid)
        batch.append(item)
        if len(batch) >= every:
            flush()
    flush()
    return {"skipped": skipped, "written": written}


In [ ]:
import json

# GENERATED from finsight.risks.teacher by scripts/make_teacher_notebook.py; do not edit
PROMPT_VERSION = 'teacher-v1'
SYSTEM = 'You label risk factors from Indian IPO offer documents and rewrite them in plain\nEnglish. Answer with one JSON object and nothing else.\n\nFields:\n- "category": exactly one of financial, debt_liquidity, customers_suppliers, competition, legal_litigation, regulatory, promoters_governance, operations, technology_data, market_macro.\n- "seriousness_1to5": 1 = routine boilerplate, 5 = a specific, material problem the company\n  already has.\n- "hard_fact": true if the risk describes something that has already happened (a loss, a case, a\n  default), false if it only describes something that may happen.\n- "simple": the risk in plain English for a first-time investor, at most 60 words, sentences under\n  20 words. Keep every number exactly as written, with its unit. Keep how certain it is: "may" stays\n  "may", "has" stays "has". Add no facts. Never tell anyone to buy, sell, apply, invest or avoid,\n  and never judge the IPO as good, bad, safe or worth it.\n- "numbers_copied": every number you used in "simple", copied exactly.\n\nCategories: financial = losses, falling revenue or margins, cash flow, working capital; debt_liquidity = borrowings, repayment, covenants, liquidity, interest rates on our loans; customers_suppliers = dependence on few customers, suppliers or distributors; competition = competitors, pricing pressure, market share; legal_litigation = court cases, claims, penalties, tax disputes; regulatory = laws, licences, approvals, government policy, SEBI/RBI rules; promoters_governance = promoters, related parties, management, control, conflicts of interest; operations = plants, capacity, people, projects, insurance, supply disruptions; technology_data = IT systems, cyber security, data protection, intellectual property; market_macro = economy, industry cycles, currency, share price after listing.'
OUTPUT_SCHEMA = json.loads('{"description": "The JSON the teacher must return for one risk.", "properties": {"category": {"enum": ["financial", "debt_liquidity", "customers_suppliers", "competition", "legal_litigation", "regulatory", "promoters_governance", "operations", "technology_data", "market_macro"], "title": "Category", "type": "string"}, "seriousness_1to5": {"maximum": 5, "minimum": 1, "title": "Seriousness 1To5", "type": "integer"}, "hard_fact": {"title": "Hard Fact", "type": "boolean"}, "simple": {"minLength": 1, "title": "Simple", "type": "string"}, "numbers_copied": {"items": {"type": "string"}, "title": "Numbers Copied", "type": "array"}}, "required": ["category", "seriousness_1to5", "hard_fact", "simple"], "title": "TeacherOutput", "type": "object"}')


def messages(title, body):
    text = f"Title: {title.strip()}\n\n{body.strip()}" if title.strip() else body.strip()
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]


In [ ]:
import json, os, shutil, time
from pathlib import Path

os.environ.setdefault("VLLM_USE_V1", "0")
import torch
from vllm import LLM, SamplingParams

ON_KAGGLE = Path("/kaggle/input").exists()
if not ON_KAGGLE:  # a dry read of the files on the laptop; vLLM itself needs the GPU
    OUT_DIR = "teacher_out"
risks_file = next(Path("/kaggle/input").rglob("risks.jsonl")) if ON_KAGGLE else Path("data/processed/teacher/risks.jsonl")
out_path = Path(OUT_DIR) / "raw.jsonl"
out_path.parent.mkdir(parents=True, exist_ok=True)
earlier = list(Path("/kaggle/input").rglob("raw.jsonl")) if ON_KAGGLE else []
if earlier and not out_path.exists():
    shutil.copy(earlier[0], out_path)  # resume
    print("resuming from", earlier[0])

risks = list(read_jsonl(risks_file))
items = [{"risk_id": r["risk_id"], "messages": messages(r.get("title") or "", r["body"])} for r in risks]
print(len(items), "risks;", len(done_ids(out_path)), "already done; GPUs:", torch.cuda.device_count())

llm = LLM(model=MODEL, quantization="awq", dtype="float16", max_model_len=MAX_MODEL_LEN,
          gpu_memory_utilization=0.92, tensor_parallel_size=max(1, torch.cuda.device_count()),
          enforce_eager=False, seed=2026)
try:  # vLLM <= 0.10
    from vllm.sampling_params import GuidedDecodingParams
    params = SamplingParams(temperature=0.2, top_p=0.9, max_tokens=MAX_TOKENS,
                            guided_decoding=GuidedDecodingParams(json=OUTPUT_SCHEMA))
except ImportError:  # newer releases renamed it
    from vllm.sampling_params import StructuredOutputsParams
    params = SamplingParams(temperature=0.2, top_p=0.9, max_tokens=MAX_TOKENS,
                            structured_outputs=StructuredOutputsParams(json=OUTPUT_SCHEMA))

def generate(chats):
    outs = llm.chat(chats, params, use_tqdm=False,
                    chat_template_kwargs={"enable_thinking": False})
    return [o.outputs[0].text for o in outs]

t0 = time.time()
stats = run_teacher(items, generate, out_path, every=EVERY, limit=LIMIT,
                    meta={"model": MODEL, "prompt_version": PROMPT_VERSION})
elapsed = time.time() - t0
summary = {**stats, "model": MODEL, "prompt_version": PROMPT_VERSION, "limit": LIMIT,
           "seconds": round(elapsed), "gpus": torch.cuda.device_count(),
           "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
(Path(OUT_DIR) / "run_summary.json").write_text(json.dumps(summary, indent=1), encoding="utf-8")
print("TEACHER OK", json.dumps(summary))
